# Gastos do governo federal com viagens (2023 a 2026)

## 1 - PERGUNTA DO NEGÓCIO

Quanto o governo federal gasta com viagens, quem gasta mais e o que mudou entre 2023 e 2026?

**Fonte:** Portal da Transparência, arquivos anuais de viagens a serviço (2023 a 2026).

**Sobre dados pessoais:** tirei nome e CPF logo na leitura dos arquivos. Quando preciso olhar por viajante, uso um número no lugar do nome.

## 2 - CONFIGURAÇÃO

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np


pd.set_option("display.max.columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

pasta_dados = Path.cwd() / "dados"

## 3 - FUNÇÃO DE CARGA E LIMPEZA

In [ ]:
## Como são quatro arquivos com o mesmo formato, criei uma função que faz a mesma limpeza em todos. ##

def carregar_e_limpar(caminho, ano):

    """Lê o CSV de viagens de um ano do Portal da Transparência e aplica a limpeza padrão"""
    df = pd.read_csv(caminho, sep=";", encoding="Windows-1252", decimal=",")

    #1 - Troca nome e CPF por uma chave temporária, que depois vira um número anônimo.
    df["chave_viajante"] = df["CPF viajante"].fillna("") + "|" + df["Nome"]
    df = df.drop(columns=["CPF viajante", "Nome"])

    #2 - Padroniza os nomes dos órgãos e dos cargos. Cargo vazio vira "NÃO INFORMADO", para não sumir das contas.
    df["Nome do órgão superior"] = (
        df["Nome do órgão superior"]
        .str.strip()
        .str.upper()
        .str.replace("MINISTÉRIO", "MIN.")
    )
    df["Cargo"] = df["Cargo"].str.strip().str.upper().fillna("NÃO INFORMADO")

    # 3. Transforma as datas de texto em data.
    for col in ["Período - Data de início", "Período - Data de fim"]:
        df[col] = pd.to_datetime(df[col], format="%d/%m/%Y", errors="coerce")

    # 4. Calcula o valor total da viagem: passagens + diárias + outros gastos − devolução.
    df["Valor Total"] = df["Valor passagens"] + df["Valor diárias"] + df["Valor outros gastos"] - df["Valor devolução"]

    # 5. Cria a coluna do ano e uma marcação para viagens com valor zero.
    df["ano"] = ano
    df["com_gasto"] = df["Valor Total"] > 0

    return df

## 4 - CARGA DOS ARQUIVOS

In [ ]:
anos = [2023, 2024, 2025, 2026]
bases = {}

for ano in anos:
    bases[ano] = carregar_e_limpar(pasta_dados / f"{ano}_Viagem.csv", ano)
    print(ano, len(bases[ano]), "linhas")

# Conferência se todos os anos estão com as mesmas colunas
colunas_ref = list(bases[2023].columns)
for ano, df_ano in bases.items():
    if list(df_ano.columns) != colunas_ref:
        print(f"Colunas de {ano} estão diferentes das de 2023", set(df_ano.columns) ^ set(colunas_ref))

#Criação de ID anônimo do viajante
viagens = pd.concat(bases.values(), ignore_index=True)
viagens["id_viajante"] = pd.factorize(viagens["chave_viajante"])[0]
viagens = viagens.drop(columns=["chave_viajante"])

del bases 
viagens.groupby("ano").size()


2023 833520 linhas
2024 800333 linhas
2025 814395 linhas
2026 461977 linhas


ano
2023    833520
2024    800333
2025    814395
2026    461977
dtype: int64

## 5 - VALIDAÇÃO

In [ ]:
## VERIFICAÇÕES GERIAS
## Antes de analisar, conferi se os dados faziam sentido: duplicadas, viagens canceladas, valores negativos e até que mês os dados de 2026 estão completos.


id_col = "Identificador do processo de viagem"

# 1. Cada viagem aparece uma vez só?
print("Viagens duplicadas:", viagens[id_col].duplicated().sum())

## INVESTIGAÇÃO DE CÉLULAS DUPLICADAS
dups = viagens[viagens[id_col].duplicated(keep=False)].sort_values(id_col)
display(dups[[id_col, "ano", "Situação", "Período - Data de início", "Valor Total"]])

# 2. Que situações existem, e quantas por ano?
display(viagens.groupby(["ano", "Situação"]).size().unstack(fill_value=0))

# 3. Existe valor total negativo?
print("Viagens com valor total negativo:", (viagens["Valor Total"] < 0).sum())

## INVESTIGAÇÃO DE VALORES NEGATIVOS
negativas = viagens[viagens["Valor Total"] < 0]
display(negativas[["ano", "Situação", "Valor diárias", "Valor passagens",
                   "Valor outros gastos", "Valor devolução", "Valor Total"]].head(10))
display(negativas["Situação"].value_counts())

# 4. % de viagens sem gasto, por ano
display(viagens.groupby("ano")["com_gasto"].value_counts(normalize=True).unstack() * 100)

# 5. Total gasto por ano (R$ milhões)
display(viagens.groupby("ano")["Valor Total"].sum() / 1e6)


Viagens duplicadas: 6


,Identificador do processo de viagem,ano,Situação,Período - Data de início,Valor Total
16718,18645681,2023,Não realizada,2023-02-13,6144.95
16719,18645681,2023,Não realizada,2023-02-13,6549.38
128200,18781059,2023,Não realizada,2023-03-24,17171.14
128201,18781059,2023,Não realizada,2023-03-24,14332.05
1573849,20598362,2024,Não realizada,2024-11-27,11585.55
1573850,20598362,2024,Não realizada,2024-11-27,13609.84
2447660,2025001682,2025,Realizada,2025-09-23,6300.68
2447661,2025001682,2025,Realizada,2025-09-08,6300.68
2447662,2025001682,2025,Realizada,2025-09-08,6300.68
2447663,2025001682,2025,Realizada,2025-09-23,6300.68


Situação,Não realizada,Realizada
ano,,
2023,16810,816710
2024,10830,789503
2025,8767,805628
2026,5378,456599


Viagens com valor total negativo: 37


,ano,Situação,Valor diárias,Valor passagens,Valor outros gastos,Valor devolução,Valor Total
10170,2023,Realizada,3593.17,0.00,0.00,3703.50,-110.33
49307,2023,Realizada,451.35,0.00,0.00,601.80,-150.45
204462,2023,Realizada,0.00,0.00,0.00,41.66,-41.66
214382,2023,Realizada,0.00,848.29,0.00,1438.20,-589.91
232372,2023,Realizada,0.00,0.00,0.00,50.75,-50.75
353995,2023,Realizada,391.53,0.00,0.00,541.98,-150.45
402975,2023,Realizada,0.00,0.00,0.00,4.68,-4.68
787435,2023,Realizada,511.61,0.00,0.00,511.90,-0.29
886859,2024,Realizada,391.53,0.00,0.00,541.98,-150.45
894923,2024,Realizada,1387.86,0.00,0.00,1890.36,-502.50


Situação
Realizada    37
Name: count, dtype: int64

com_gasto,False,True
ano,,
2023,5.37,94.63
2024,4.74,95.26
2025,5.30,94.70
2026,5.98,94.02


ano
2023   2308.38
2024   2413.28
2025   2520.51
2026   1329.70
Name: Valor Total, dtype: float64

In [ ]:
## INVESTIGAÇÃO DOS MESES DE 2026

v26 = viagens[viagens["ano"] == 2026]
display(v26["Período - Data de início"].dt.to_period("M").value_counts().sort_index())

Período - Data de início
2026-01    27962
2026-02    43512
2026-03    75989
2026-04    64818
2026-05    86926
2026-06    74366
2026-07    42817
2026-08    44967
2026-09      588
2026-10       22
2026-11        9
2026-12        1
Freq: M, Name: count, dtype: int64

### 5.1 - Decisões da validação

- **Tirei as viagens "Não realizada"** (de 1% a 2% por ano), porque não viraram gasto de verdade.
- **Tirei só as linhas repetidas por inteiro** (3 casos). Quando o mesmo número de viagem aparece com datas diferentes, mantive: são trechos diferentes da mesma viagem.
- **Mantive as 37 viagens com valor negativo.** Nelas, a devolução foi maior que o gasto. São valores pequenos e reais.
- **Os dados de 2026 vão até agosto, mas julho e agosto ainda estão incompletos.** Por isso, quando comparo os anos, uso janeiro a junho.

In [ ]:
antes = len(viagens)
viagens = viagens[viagens["Situação"] == "Realizada"]  # remove "Não realizada"
viagens = viagens.drop_duplicates()                    # remove linha 100% idêntica
print(f"Linhas removidas: {antes - len(viagens):,}")

# Mês de início, para comparar os mesmos meses entre os anos
viagens["mes_inicio"] = viagens["Período - Data de início"].dt.month

Linhas removidas: 41,785


## 6 - ANÁLISES

### 6.1 - Quanto se gastou em cada ano

Gasto total, número de viagens e custo médio por viagem. 2026 vai só até junho, então não dá para comparar o valor dele com os outros anos (isso fica para a 6.2).

In [ ]:
base_ano = viagens[
    (viagens["Período - Data de início"].dt.year == viagens["ano"])
    & ((viagens["ano"] < 2026) | (viagens["mes_inicio"] <= 6))
]

resumo_ano = base_ano.groupby("ano").agg(
    gasto_total_mi=("Valor Total", lambda x: x.sum() / 1e6),
    viagens=("Valor Total", "size"),
)

# Custo médio só das viagens com gasto (sem as de valor zero)
resumo_ano["custo_medio_viagem"] = (
    base_ano[base_ano["com_gasto"]].groupby("ano")["Valor Total"].mean()
)

display(resumo_ano.round(1))

,gasto_total_mi,viagens,custo_medio_viagem
ano,,,
2023,2278.70,816710,2920.70
2024,2377.80,789503,3161.30
2025,2491.30,805628,3267.20
2026,1088.60,369435,3132.70


**Conclusão 6.1:** o gasto foi de R$ 2,28 bi em 2023, R$ 2,38 bi em 2024 e R$ 2,49 bi em 2025, uma alta de 9,3% em dois anos. Mas o número de viagens até caiu um pouco (−1,4%). O que subiu foi o custo de cada viagem: de R$ 2.921 para R$ 3.267 (+11,9%). Em 2026, até junho, o custo médio está em R$ 3.133, mas o ano ainda não fechou.

### 6.2 - Comparando os anos no mesmo período

Para comparar 2026 com os outros anos, usei só os mesmos meses em todos eles.

In [ ]:
# Jan - Ago

jan_ago = viagens[
    (viagens["Período - Data de início"].dt.year == viagens["ano"])
    & (viagens["mes_inicio"] <= 8)
]
gasto_jan_ago = (jan_ago.groupby("ano")["Valor Total"].sum() / 1e6).round(1)
display(gasto_jan_ago   )

# Variação em relação a 2023
display(((gasto_jan_ago / gasto_jan_ago[2023] - 1) * 100).round(1))

ano
2023   1328.40
2024   1534.20
2025   1551.30
2026   1308.90
Name: Valor Total, dtype: float64

ano
2023    0.00
2024   15.50
2025   16.80
2026   -1.50
Name: Valor Total, dtype: float64

In [ ]:
# Jan - Jun

jan_jun = viagens[
    (viagens["Período - Data de início"].dt.year == viagens["ano"])
    & (viagens["mes_inicio"] <= 6)
]
gasto_jan_jun = (jan_jun.groupby("ano")["Valor Total"].sum() / 1e6).round(1)
display(gasto_jan_jun)

# Variação em relação a 2023
display(((gasto_jan_jun / gasto_jan_jun[2023] - 1) * 100).round(1))

ano
2023    949.20
2024   1150.40
2025   1180.50
2026   1088.60
Name: Valor Total, dtype: float64

ano
2023    0.00
2024   21.20
2025   24.40
2026   14.70
Name: Valor Total, dtype: float64

**Conclusão 6.2:** de janeiro a junho, o gasto subiu de R$ 949 mi (2023) para R$ 1.181 mi (2025), alta de 24%. Em 2026, caiu para R$ 1.089 mi, 7,8% abaixo de 2025.

Primeiro comparei de janeiro a agosto, e a queda parecia de 15,6%. Mas quase toda a diferença estava em julho e agosto, que caíram 41% de um ano para o outro. Isso é sinal de que esses meses ainda não foram totalmente lançados. Por isso, considero mais confiável a queda de cerca de 8%, e vale rever quando o Portal atualizar os dados.

### 6.3 - Quais órgãos gastam mais

Quantos órgãos somam 80% do gasto e quais são os maiores. 

Uso o ano completo de 2023 a 2025 e janeiro a junho de 2026.

Como comparo porcentagens, cada ano é dividido pelo seu próprio total.

#### PARTE 1 - PARETO DE CADA ANO

In [ ]:
gasto_orgao = (
    base_ano.groupby(["ano", "Nome do órgão superior"])["Valor Total"]
    .sum()
    .reset_index()
)

def resumo_pareto(df_ano):
    df_ano = df_ano.sort_values("Valor Total", ascending=False)
    pct = df_ano["Valor Total"] / df_ano["Valor Total"].sum() * 100
    acumulado = pct.cumsum()
    return pd.Series({
        "órgãos no total": len(df_ano),
        "órgãos para 80% do gasto": (acumulado < 80).sum() + 1,
        "% do maior órgão": round(pct.iloc[0], 1),
    })

display(
    gasto_orgao.groupby("ano").apply(resumo_pareto)
    .astype({"órgãos no total": int, "órgãos para 80% do gasto": int})
)




,órgãos no total,órgãos para 80% do gasto,% do maior órgão
ano,,,
2023,35,9,34.20
2024,35,10,33.10
2025,35,10,33.00
2026,35,12,26.90


#### Parte 2 - 5 maiortes órgãos por ano (%)

In [ ]:
gasto_orgao["% do ano"] = (
    gasto_orgao["Valor Total"]
    / gasto_orgao.groupby("ano")["Valor Total"].transform("sum") * 100
)
gasto_orgao["R$ mi"] = gasto_orgao["Valor Total"] / 1e6

top5 = (
    gasto_orgao.sort_values(["ano", "Valor Total"], ascending=[True, False])
    .groupby("ano")
    .head(5)
)

tabela_top5 = (
    top5.pivot(index="Nome do órgão superior", columns="ano", values=["R$ mi", "% do ano"])
    .swaplevel(axis=1)       # agrupa as colunas por ano
    .sort_index(axis=1)      # 2023, 2024... com R$ mi e % lado a lado
    .round(1)
)   
display(tabela_top5)

ano                                          2023            2024         \
                                         % do ano  R$ mi % do ano  R$ mi   
Nome do órgão superior                                                     
MIN. DA DEFESA                              13.90 316.20    14.10 334.40   
MIN. DA EDUCAÇÃO                            11.90 270.60    11.90 284.00   
MIN. DA JUSTIÇA E SEGURANÇA PÚBLICA         34.20 779.20    33.10 788.20   
MIN. DA SAÚDE                                3.70  84.50     3.70  87.70   
MIN. DO MEIO AMBIENTE E MUDANÇA DO CLIMA     5.00 113.00     5.20 123.40   

ano                                          2025            2026         
                                         % do ano  R$ mi % do ano  R$ mi  
Nome do órgão superior                                                    
MIN. DA DEFESA                              13.80 344.80    12.60 137.10  
MIN. DA EDUCAÇÃO                            13.10 325.60    12.80 139.40  
MIN. DA JUSTIÇA E SEGURANÇA PÚBLICA         33.00 821.10    26.90 292.50  
MIN. DA SAÚDE                                3.40  84.00     4.50  48.70  
MIN. DO MEIO AMBIENTE E MUDANÇA DO CLIMA     5.30 132.70     4.80  52.20

**Conclusão 6.3:** o gasto fica em poucas mãos. De 35 órgãos, 9 somavam 80% do gasto em 2023, e 10 em 2024 e 2025. Os mesmos 5 órgãos lideram todos os anos e ficam com cerca de 69% do total. Só o Ministério da Justiça e Segurança Pública responde por um terço (R$ 821 mi em 2025). Entre os grandes, a Educação foi a que mais cresceu: +20% de 2023 a 2025.

Em 2026, até junho, foram precisos 12 órgãos para chegar a 80%, e o peso do Ministério da Justiça caiu para 26,9%. Como os dados de 2026 ainda estão incompletos, prefiro esperar o ano fechar para confirmar.

### 6.4 - Quais cargos gastam mais

Muitas viagens não têm o cargo do viajante: ou está em sigilo, ou não foi informado. Primeiro vejo quanto do gasto tem cargo, depois quais cargos gastam mais.

#### Parte 1 - Quanto dos gastos tem cargo identificado?

In [ ]:
base_ano["categoria_cargo"] = np.select(
    [
        base_ano["Cargo"].eq("INFORMAÇÕES PROTEGIDAS POR SIGILO"),
        base_ano["Cargo"].eq("NÃO INFORMADO"),
    ],
    ["Sigilo", "Não informado"],
    default="Identificado",
)

pct_gasto_categoria = (
    base_ano.groupby(["ano", "categoria_cargo"])["Valor Total"].sum()
    / base_ano.groupby("ano")["Valor Total"].sum() * 100
)
display(pct_gasto_categoria.unstack().round(1))

categoria_cargo,Identificado,Não informado,Sigilo
ano,,,
2023,38.00,45.70,16.30
2024,37.30,45.60,17.00
2025,37.50,45.50,17.00
2026,42.00,39.60,18.40


Só cerca de 38% do gasto tem cargo identificado. O resto é de viagens sem cargo (cerca de 45%) ou em sigilo (cerca de 17%). Por isso, a parte 2 considera só os cargos identificados.

#### De onde vem o gasto sem cargo

In [ ]:
# Gasto sem cargo por órgão, em % do gasto sem cargo de cada ano
sem_cargo = base_ano[base_ano["categoria_cargo"] == "Não informado"]

pct_sem_cargo = (
    sem_cargo.groupby(["ano", "Nome do órgão superior"])["Valor Total"].sum()
    / sem_cargo.groupby("ano")["Valor Total"].sum() * 100
).unstack("ano")

display(pct_sem_cargo.sort_values(2025, ascending=False).head(5).round(1))

# Funções mais comuns entre as viagens sem cargo (todos os anos)
display(sem_cargo["Descrição Função"].value_counts().head(5))

ano,2023,2024,2025,2026
Nome do órgão superior,,,,
MIN. DA JUSTIÇA E SEGURANÇA PÚBLICA,39.60,34.90,34.30,21.40
MIN. DA DEFESA,29.60,30.20,29.80,31.20
MIN. DA EDUCAÇÃO,4.60,5.20,6.50,9.00
MIN. DO MEIO AMBIENTE E MUDANÇA DO CLIMA,4.00,5.40,4.50,3.70
PRESIDÊNCIA DA REPÚBLICA,3.70,4.10,3.10,4.70


Descrição Função
Sem informação    509404
Sargentos         164761
OfSuperior         72935
Praças             65304
OfSubalter         52420
Name: count, dtype: int64

Nos quatro anos, a maior parte do gasto sem cargo é do Ministério da Justiça e Segurança Pública e da Defesa: juntos, eram 69% em 2023 e 64% em 2025. A Defesa fica estável, em cerca de 30%, e a coluna de função mostra patentes (sargentos, oficiais, praças): os militares aparecem pela patente, não por um cargo. Já o Ministério da Justiça cai ano a ano, de 40% em 2023 para 34% em 2025 e 21% em 2026, acompanhando a queda do peso dele no gasto total (6.3).

   #### Parte 2 - os 10 cargos que mais gastaram

In [ ]:
def top_cargos_ano(ano, n=10):
    base = base_ano[
        (base_ano["ano"] == ano) & (base_ano["categoria_cargo"] == "Identificado")
    ]
    tabela = base.groupby("Cargo").agg(
        gasto_mi=("Valor Total", lambda x: x.sum() / 1e6),
        viagens=("Valor Total", "size"),
    )
    tabela["custo_medio"] = tabela["gasto_mi"] * 1e6 / tabela["viagens"]
    tabela["% do gasto identificado"] = (
        tabela["gasto_mi"] / tabela["gasto_mi"].sum() * 100
    )
    return tabela.sort_values("gasto_mi", ascending=False).head(n).round(1)

for ano in [2023, 2024, 2025, 2026]:
    titulo = f"{ano} (janeiro a junho, parcial)" if ano == 2026 else str(ano)
    print(f"\nTop 10 cargos por gasto: {titulo}")
    display(top_cargos_ano(ano))


Top 10 cargos por gasto: 2023


,gasto_mi,viagens,custo_medio,% do gasto identificado
Cargo,,,,
PROFESSOR DO MAGISTERIO SUPERIOR,108.80,53399,2038.00,12.60
PROFESSOR ENS BASICO TECN TECNOLOGICO,52.20,37172,1404.40,6.00
TECNICO DO SEGURO SOCIAL,45.60,10542,4323.90,5.30
ANALISTA AMBIENTAL,34.50,13254,2604.80,4.00
AUDITOR FISCAL FEDERAL AGROPECUARIO,23.80,16088,1480.30,2.80
ESP EM REGULACAO E VIGILANCIA SANITARIA,23.80,1806,13165.40,2.70
AUDITOR-FISCAL DA RECEITA FEDERAL BRASIL,22.70,9421,2407.10,2.60
CONTRATADO LEI 8745/93 - NI,17.20,17584,978.70,2.00
TECNICO AMBIENTAL,17.10,5839,2929.90,2.00



Top 10 cargos por gasto: 2024


,gasto_mi,viagens,custo_medio,% do gasto identificado
Cargo,,,,
PROFESSOR DO MAGISTERIO SUPERIOR,114.40,52135,2193.60,12.90
PROFESSOR ENS BASICO TECN TECNOLOGICO,53.20,35355,1504.30,6.00
TECNICO DO SEGURO SOCIAL,44.50,9457,4708.70,5.00
ANALISTA AMBIENTAL,35.60,12804,2783.80,4.00
AUDITOR-FISCAL DA RECEITA FEDERAL BRASIL,30.40,10612,2866.70,3.40
ESP EM REGULACAO E VIGILANCIA SANITARIA,26.70,1815,14684.70,3.00
AUDITOR FISCAL FEDERAL AGROPECUARIO,24.40,16040,1522.90,2.80
ANALISTA TRIBUTARIO REC FEDERAL BRASIL,19.80,9785,2019.70,2.20
ASSISTENTE EM ADMINISTRAÇÃO,15.80,10528,1498.90,1.80



Top 10 cargos por gasto: 2025


,gasto_mi,viagens,custo_medio,% do gasto identificado
Cargo,,,,
PROFESSOR DO MAGISTERIO SUPERIOR,119.50,54353,2198.30,12.80
PROFESSOR ENS BASICO TECN TECNOLOGICO,62.90,40294,1561.60,6.70
TECNICO DO SEGURO SOCIAL,50.30,10059,4996.80,5.40
ANALISTA AMBIENTAL,48.50,16449,2947.70,5.20
ESP EM REGULACAO E VIGILANCIA SANITARIA,25.70,1896,13533.50,2.70
AUDITOR-FISCAL DA RECEITA FEDERAL BRASIL,25.00,8663,2889.80,2.70
AUDITOR FISCAL FEDERAL AGROPECUARIO,24.90,17151,1453.60,2.70
TECNICO AMBIENTAL,17.30,5737,3022.70,1.90
ASSISTENTE EM ADMINISTRAÇÃO,16.90,11256,1505.60,1.80



Top 10 cargos por gasto: 2026 (janeiro a junho, parcial)


,gasto_mi,viagens,custo_medio,% do gasto identificado
Cargo,,,,
PROFESSOR DO MAGISTERIO SUPERIOR,47.20,19878,2375.80,10.30
TECNICO DO SEGURO SOCIAL,28.60,6724,4252.10,6.30
PROFESSOR ENS BASICO TECN TECNOLOGICO,27.20,16407,1660.20,6.00
ANALISTA AMBIENTAL,23.00,8655,2662.40,5.00
AUDITOR-FISCAL DA RECEITA FEDERAL BRASIL,18.70,6838,2741.20,4.10
ESP EM REGULACAO E VIGILANCIA SANITARIA,11.90,1009,11787.00,2.60
AUDITOR FISCAL FEDERAL AGROPECUARIO,11.80,8320,1414.70,2.60
ANALISTA TRIBUTARIO REC FEDERAL BRASIL,10.10,4684,2148.10,2.20
AGENTE ADMINISTRATIVO,9.30,2719,3436.60,2.00


**Conclusão 6.4:** a análise por cargo cobre só cerca de 38% do gasto. O resto é de viagens sem cargo (cerca de 45%) ou em sigilo (cerca de 17%). Boa parte do que fica de fora é do Ministério da Justiça e Segurança Pública e da Defesa: juntos, eram 69% do gasto sem cargo em 2023 e 64% em 2025. Na Defesa, são militares, que aparecem pela patente e não por um cargo.

Nos cargos identificados, quase nada muda de um ano para o outro. O Professor do Magistério Superior está em primeiro todos os anos (R$ 119,5 mi em 2025, cerca de 13% do gasto identificado). O Analista Ambiental foi o que mais cresceu: +41% de 2023 a 2025, com mais viagens (+24%) e viagens mais caras (+13%). E o custo médio subiu em quase todos os cargos do topo, o que confirma a 6.1: o gasto aumentou porque cada viagem ficou mais cara.

Um caso que chama atenção: o Especialista em Regulação e Vigilância Sanitária faz poucas viagens (cerca de 1.900 por ano), mas cada uma custa em média R$ 13.500, cerca de 4 vezes a média geral.

### 6.5 - Para onde vão as viagens

Quais os destinos mais frequentes, onde gasta mais e quantos são para fora do país.

Antes, confiro quantos destinos estão em sigilo e como aparecem as viagens com mais de uma cidade.

#### Parte 1: quantos destinos dá para identificar

In [ ]:
base_ano["categoria_destino"] = np.select(
    [
        base_ano["Destinos"].eq("Informações protegidas por sigilo"),
        base_ano["Destinos"].eq("Sem informação") | base_ano["Destinos"].isna(),
    ],
    ["Sigilo", "Sem informação"],
    default="Identificado",
)

display(
    (base_ano.groupby("ano")["categoria_destino"].value_counts(normalize=True) * 100)
    .unstack().round(1)
)

categoria_destino,Identificado,Sem informação,Sigilo
ano,,,
2023,85.30,0.40,14.40
2024,84.70,0.30,15.00
2025,85.60,0.30,14.00
2026,85.10,0.30,14.60


Cerca de 85% das viagens têm destino identificado. Os outros 15% estão quase todos em sigilo, e essa proporção quase não muda de um ano para o outro.

#### Parte 2: como aparecem as viagens com mais de uma cidade

In [ ]:
identificados = base_ano[base_ano["categoria_destino"] == "Identificado"]

# Quantas "/" cada destino tem: 1 = uma cidade, 2 ou mais = várias cidades
qtd_barras = identificados["Destinos"].str.count("/")
display(qtd_barras.value_counts().sort_index().head(6))

# Exemplos de viagens com mais de um destino
display(identificados.loc[qtd_barras > 1, "Destinos"].head(5).tolist())

Destinos
1    2083825
2     162422
3      62597
4      26216
5      13142
6       6387
Name: count, dtype: int64

['Pelotas/RS, Pelotas/RS',
 'Brasília/DF, Brasília/DF',
 'Brasília/DF, São João D´Aliança/GO, Brasília/DF',
 'Brasília/DF, São João D´Aliança/GO, Brasília/DF',
 'Montevidéu/Uruguai, Buenos Aires/Argentina']

88% das viagens identificadas vão para uma cidade só. As outras passam por duas ou mais, separadas por vírgula. Às vezes a mesma cidade aparece mais de uma vez no roteiro (por exemplo, "Brasília/DF, São João D'Aliança/GO, Brasília/DF"). Esse arquivo não mostra o porquê, então conto cada cidade uma vez por viagem.

#### Parte 3: destinos com mais viagens e mais gasto

In [ ]:
UFS = {"AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB",
       "PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"}

UFS = {"AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB",
       "PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"}

# 1. Separa as cidades de cada viagem, sem repetir a mesma cidade
dest = identificados[["ano", "Valor Total", "Destinos"]].copy()
dest["cidades"] = dest["Destinos"].str.split(", ").apply(lambda l: list(dict.fromkeys(l)))
dest["valor_por_cidade"] = dest["Valor Total"] / dest["cidades"].str.len()

# 2. Uma linha por cidade visitada
por_cidade = dest.explode("cidades")
por_cidade["internacional"] = ~por_cidade["cidades"].str.split("/").str[-1].isin(UFS)

# 3. Viagens e gasto por cidade, em cada ano
cid_ano = por_cidade.groupby(["ano", "cidades"]).agg(
    viagens=("cidades", "size"),
    gasto_mi=("valor_por_cidade", lambda x: x.sum() / 1e6),
).reset_index()

def tabela_cidades(ordenar_por, n=10):
    # As n cidades com mais viagens (ou gasto) somando 2023 a 2025
    top = (
        cid_ano[cid_ano["ano"] < 2026]
        .groupby("cidades")[ordenar_por].sum()
        .nlargest(n).index
    )
    return (
        cid_ano[cid_ano["cidades"].isin(top)]
        .pivot(index="cidades", columns="ano", values=["viagens", "gasto_mi"])
        .swaplevel(axis=1).sort_index(axis=1)
        .reindex(top)
        .round(1)
    )

print("10 cidades com mais viagens (2026 = janeiro a junho)")
display(tabela_cidades("viagens"))

print("10 cidades com mais gasto (2026 = janeiro a junho)")
display(tabela_cidades("gasto_mi"))

# 4. Peso das viagens internacionais, por ano
dest["internacional"] = por_cidade.groupby(level=0)["internacional"].any()

resumo_intl = dest.groupby("ano").agg(
    pct_viagens_internacionais=("internacional", lambda x: x.mean() * 100),
)
resumo_intl["pct_gasto_internacionais"] = (
    dest[dest["internacional"]].groupby("ano")["Valor Total"].sum()
    / dest.groupby("ano")["Valor Total"].sum() * 100
)
display(resumo_intl.round(1))

# Viagens internacionais: os 10 países com mais gasto
internacionais = por_cidade[por_cidade["internacional"]].copy()
internacionais["pais"] = internacionais["cidades"].str.split("/").str[-1]

pais_ano = internacionais.groupby(["ano", "pais"]).agg(
    viagens=("pais", "size"),
    gasto_mi=("valor_por_cidade", lambda x: x.sum() / 1e6),
).reset_index()

# Escolho os 10 países pela soma do gasto de 2023 a 2025 (anos completos); a tabela mostra os 4 anos
top_paises = (
    pais_ano[pais_ano["ano"] < 2026]
    .groupby("pais")["gasto_mi"].sum()
    .nlargest(10).index
)

print("10 países com mais gasto em viagens internacionais, ordenados pelo gasto de 2023 a 2025 (2026 = janeiro a junho)")
display(
    pais_ano[pais_ano["pais"].isin(top_paises)]
    .pivot(index="pais", columns="ano", values=["viagens", "gasto_mi"])
    .swaplevel(axis=1).sort_index(axis=1)
    .reindex(top_paises)
    .round(1)
)

10 cidades com mais viagens (2026 = janeiro a junho)


ano                        2023              2024              2025           \
                       gasto_mi  viagens gasto_mi  viagens gasto_mi  viagens   
cidades                                                                        
Brasília/DF              321.20 69029.00   320.10 65488.00   359.90 70843.00   
Rio de Janeiro/RJ         86.80 29822.00   111.30 30718.00    99.30 28887.00   
São Paulo/SP              67.30 27190.00    82.40 31694.00    79.40 29796.00   
São José dos Campos/SP    26.60 12846.00    28.10 11893.00    35.70 13430.00   
Belo Horizonte/MG         20.70  9608.00    23.70 10270.00    28.20 11353.00   
Manaus/AM                 33.20  9856.00    36.10  9862.00    44.40 11099.00   
Recife/PE                 22.40  9484.00    28.10 10849.00    28.00 10097.00   
Salvador/BA               24.90 10010.00    24.60  9226.00    26.90  9213.00   
Curitiba/PR               22.00  9741.00    21.10  8978.00    24.80  9668.00   
Belém/PA                  27.70  8233.00    30.20  8996.00    47.30 10653.00   

ano                        2026           
                       gasto_mi  viagens  
cidades                                   
Brasília/DF              166.60 36391.00  
Rio de Janeiro/RJ         43.90 12282.00  
São Paulo/SP              44.00 14955.00  
São José dos Campos/SP    12.30  5179.00  
Belo Horizonte/MG         12.00  4915.00  
Manaus/AM                 17.00  4897.00  
Recife/PE                 12.60  4118.00  
Salvador/BA               15.10  5050.00  
Curitiba/PR               10.80  4323.00  
Belém/PA                  13.80  3799.00

10 cidades com mais gasto (2026 = janeiro a junho)


ano                        2023              2024              2025           \
                       gasto_mi  viagens gasto_mi  viagens gasto_mi  viagens   
cidades                                                                        
Brasília/DF              321.20 69029.00   320.10 65488.00   359.90 70843.00   
Rio de Janeiro/RJ         86.80 29822.00   111.30 30718.00    99.30 28887.00   
São Paulo/SP              67.30 27190.00    82.40 31694.00    79.40 29796.00   
Manaus/AM                 33.20  9856.00    36.10  9862.00    44.40 11099.00   
Belém/PA                  27.70  8233.00    30.20  8996.00    47.30 10653.00   
Boa Vista/RR              27.30  6741.00    35.10  6902.00    39.20  7407.00   
São José dos Campos/SP    26.60 12846.00    28.10 11893.00    35.70 13430.00   
Foz do Iguaçu/PR          28.60  7180.00    30.40  7231.00    30.10  7143.00   
Fortaleza/CE              22.80  7755.00    30.90  9743.00    31.50  9421.00   
Recife/PE                 22.40  9484.00    28.10 10849.00    28.00 10097.00   

ano                        2026           
                       gasto_mi  viagens  
cidades                                   
Brasília/DF              166.60 36391.00  
Rio de Janeiro/RJ         43.90 12282.00  
São Paulo/SP              44.00 14955.00  
Manaus/AM                 17.00  4897.00  
Belém/PA                  13.80  3799.00  
Boa Vista/RR               8.50  2112.00  
São José dos Campos/SP    12.30  5179.00  
Foz do Iguaçu/PR          12.00  2932.00  
Fortaleza/CE              15.90  4149.00  
Recife/PE                 12.60  4118.00

,pct_viagens_internacionais,pct_gasto_internacionais
ano,,
2023,3.00,15.10
2024,3.00,14.40
2025,2.70,12.70
2026,2.60,14.30


10 países com mais gasto em viagens internacionais, ordenados pelo gasto de 2023 a 2025 (2026 = janeiro a junho)


ano                           2023             2024             2025          \
                          gasto_mi viagens gasto_mi viagens gasto_mi viagens   
pais                                                                           
Estados Unidos da América    53.10 3739.00    57.30 3881.00    41.20 2761.00   
França                       14.50 1159.00    14.20  999.00    18.60 1304.00   
China                        13.10  812.00    14.50  799.00    17.30 1076.00   
Portugal                     15.90 1537.00    15.00 1400.00    12.20 1157.00   
Suíça                        15.10  676.00    11.00  557.00    12.50  547.00   
Alemanha                      9.90  795.00     9.30  689.00    10.30  683.00   
Espanha                       8.40  840.00     9.60  831.00    10.70  843.00   
Índia                        12.20  592.00     7.40  338.00     9.00  404.00   
Itália                        8.70  723.00    10.40  847.00     7.70  713.00   
Reino Unido                   6.60  597.00     6.20  547.00     8.40  557.00   

ano                           2026          
                          gasto_mi viagens  
pais                                        
Estados Unidos da América    22.30 1432.00  
França                        9.50  661.00  
China                        11.70  538.00  
Portugal                      6.20  506.00  
Suíça                         6.10  254.00  
Alemanha                      5.80  372.00  
Espanha                       6.70  488.00  
Índia                         6.20  268.00  
Itália                        3.20  265.00  
Reino Unido                   5.60  377.00

**Conclusão 6.5:** Brasília é de longe o destino principal: cerca de 70 mil viagens por ano e R$ 360 mi em 2025, quase 4 vezes o Rio de Janeiro, que vem em segundo. Sozinha, Brasília fica com cerca de 14% de todo o gasto com viagens.

As cidades com mais viagens não são as mesmas com mais gasto. Boa Vista, Foz do Iguaçu e Fortaleza aparecem entre as que mais custam, mesmo sem estar entre as mais visitadas. Isso acontece porque cada viagem para lá é mais cara. Em 2025, uma viagem para Boa Vista custou em média R$ 5.290, e para Brasília R$ 5.080, contra R$ 2.660 para São Paulo. Belém e Manaus foram as que mais cresceram: +71% e +34% no gasto de 2023 a 2025.

As viagens internacionais são poucas, cerca de 3% das viagens com destino identificado, mas ficam com 13% a 15% do gasto. Na média, cada uma custa cerca de 5 vezes uma viagem nacional. Os Estados Unidos são o principal destino fora do Brasil, mas o gasto lá caiu 28% em 2025 (de R$ 57 mi para R$ 41 mi), com cerca de 29% menos viagens. Cada viagem para os Estados Unidos custou, em média, cerca de R$ 15 mil em 2025. A China foi o país que mais cresceu: +32% no gasto de 2023 a 2025, e de 812 para 1.076 viagens.

Essa análise cobre os 85% das viagens com destino identificado. Nas viagens com mais de uma cidade, dividi o valor igualmente entre elas.

### 6.6 - Viagens urgentes

Quero ver quantas viagens são marcadas como urgentes, se elas custam mais que as outras e em que órgãos isso acontece mais.

#### Parte 1: quanto das viagens e do gasto é urgente

In [ ]:
base_ano["urgente"] = base_ano["Viagem Urgente"].eq("SIM")

urgencia_ano = base_ano.groupby("ano").agg(
    pct_viagens_urgentes=("urgente", lambda x: x.mean() * 100),
)
urgencia_ano["pct_gasto_urgente"] = (
    base_ano[base_ano["urgente"]].groupby("ano")["Valor Total"].sum()
    / base_ano.groupby("ano")["Valor Total"].sum() * 100
)
display(urgencia_ano.round(1))

,pct_viagens_urgentes,pct_gasto_urgente
ano,,
2023,62.50,64.80
2024,58.70,61.30
2025,57.60,60.40
2026,58.50,59.80


Eu esperava poucas viagens urgentes, mas são quase 60% na média dos anos. Se a maioria é urgente, a marcação provavelmente não quer dizer emergência: o mais provável é que indique viagens pedidas com menos antecedência do que o mínimo exigido. A % caiu de 62,5% em 2023 para 57,6% em 2025. O peso delas no gasto (60,4% em 2025) é quase igual ao peso no número de viagens.

#### Parte 2: uma viagem urgente custa mais?

In [ ]:
# Só viagens com gasto, para o valor zero não puxar a média para baixo
com_gasto = base_ano[base_ano["com_gasto"]]

custo_urgencia = (
    com_gasto.groupby(["ano", "urgente"])[["Valor Total", "Valor passagens"]]
    .mean()
    .unstack("urgente")
    .round(0)
)
custo_urgencia.columns = [
    f"{'urgente' if urg else 'normal'} - {col.lower()}"
    for col, urg in custo_urgencia.columns
]
display(custo_urgencia)

,normal - valor total,urgente - valor total,normal - valor passagens,urgente - valor passagens
ano,,,,
2023,2977.00,2891.00,1134.00,1023.00
2024,3183.00,3148.00,1099.00,1056.00
2025,3309.00,3241.00,1162.00,1137.00
2026,3330.00,3013.00,1184.00,1402.00


Considerando só as viagens com gasto, a urgente custa, na média, um pouco menos que a normal: R$ 3.241 contra R$ 3.309 em 2025. A passagem também: R$ 1.137 contra R$ 1.162. Ou seja, ser urgente não deixou a viagem mais cara, o que reforça que a marcação tem mais a ver com prazo do que com emergência. A exceção é 2026, em que a passagem urgente está 18% mais cara, mas o ano ainda está incompleto.

Uma observação: essa média mistura destinos e durações diferentes. Para comparar de forma justa, o ideal seria olhar o mesmo destino.

#### Parte 3: em que órgãos há mais viagens urgentes

In [ ]:
urg_orgao = base_ano.groupby(["ano", "Nome do órgão superior"]).agg(
    viagens=("urgente", "size"),
    urgentes=("urgente", "sum"),
).reset_index()
urg_orgao["% urgentes"] = urg_orgao["urgentes"] / urg_orgao["viagens"] * 100

# Escolho os 10 órgãos com mais viagens urgentes de 2023 a 2025 (anos completos)
top_urg = (
    urg_orgao[urg_orgao["ano"] < 2026]
    .groupby("Nome do órgão superior")["urgentes"].sum()
    .nlargest(10).index
)

print("10 órgãos com mais viagens urgentes: % das viagens do órgão que foram urgentes (2026 = janeiro a junho)")
display(
    urg_orgao[urg_orgao["Nome do órgão superior"].isin(top_urg)]
    .pivot(index="Nome do órgão superior", columns="ano", values="% urgentes")
    .reindex(top_urg)
    .round(1)
)

10 órgãos com mais viagens urgentes: % das viagens do órgão que foram urgentes (2026 = janeiro a junho)


ano,2023,2024,2025,2026
Nome do órgão superior,,,,
MIN. DA EDUCAÇÃO,69.60,67.30,67.30,72.30
MIN. DA DEFESA,76.60,73.50,72.20,76.20
MIN. DA JUSTIÇA E SEGURANÇA PÚBLICA,41.50,30.70,23.60,25.90
MIN. DO PLANEJAMENTO E ORÇAMENTO,79.70,76.70,80.40,78.90
MIN. DO MEIO AMBIENTE E MUDANÇA DO CLIMA,61.80,57.00,55.10,46.20
MIN. DO DESENVOLVIMENTO AGRÁRIO E AGRICULTURA FAMILIAR,85.40,77.10,77.70,83.50
MIN. DA FAZENDA,65.60,57.00,54.50,50.30
MIN. DA AGRICULTURA E PECUÁRIA,54.90,55.10,57.80,62.50
MIN. DA SAÚDE,61.40,60.00,62.60,70.50


Na maioria dos órgãos, mais da metade das viagens é urgente. O Ministério da Justiça é a grande exceção: caiu de 41,5% em 2023 para 23,6% em 2025. Como é o órgão que mais gasta com viagens, essa queda deve explicar boa parte da queda geral. Os órgãos com mais urgência são Desenvolvimento Agrário (de 77% a 85%), Planejamento (de 77% a 80%) e Defesa (de 72% a 77%). Meio Ambiente e Fazenda vêm reduzindo a urgência, enquanto Previdência e Saúde estão aumentando.

**Conclusão 6.6:** quase 60% das viagens são marcadas como urgentes. Com tanta viagem urgente, a marcação provavelmente indica pedido feito com pouca antecedência, e não emergência. Isso também aparece no custo: na média, a viagem urgente não custa mais que a normal (R$ 3.241 contra R$ 3.309 em 2025), nem a passagem.

O % de urgentes caiu de 62,5% em 2023 para 57,6% em 2025, puxado principalmente pelo Ministério da Justiça, o órgão que mais gasta e o que mais reduziu a urgência (de 41,5% para 23,6%). Na maioria dos outros órgãos, a urgência continua acima de 55%, e em alguns, como Previdência e Saúde, está aumentando.

### 6.7 - Por que cada viagem ficou mais cara

No item 6.1 notei que o gasto subiu porque o custo por viagem aumentou. 

Agora quero saber o que exatamente ficou mais caro, passagens, diárias ou outros gastos.

#### Parte 1: o custo médio por componente

In [ ]:
componentes = ["Valor passagens", "Valor diárias", "Valor outros gastos", "Valor devolução"]

# Média por viagem, só das viagens com gasto
custo_comp = com_gasto.groupby("ano")[componentes + ["Valor Total"]].mean()
display(custo_comp.round(0))

# Quanto cada componente contribuiu para o aumento de 2023 para 2025
variacao = custo_comp.loc[2025] - custo_comp.loc[2023]
variacao["Valor devolução"] = -variacao["Valor devolução"]  # devolução maior reduz o custo

contribuicao = pd.DataFrame({
    "2023 (R$)": custo_comp.loc[2023],
    "2025 (R$)": custo_comp.loc[2025],
    "variação (R$)": variacao,
    "% do aumento total": variacao / variacao["Valor Total"] * 100,
})
display(contribuicao.round(1))

,Valor passagens,Valor diárias,Valor outros gastos,Valor devolução,Valor Total
ano,,,,,
2023,1061.00,1857.00,15.00,13.00,2921.00
2024,1073.00,2086.00,16.00,15.00,3161.00
2025,1146.00,2118.00,15.00,12.00,3267.00
2026,1320.00,1804.00,19.00,10.00,3133.00


,2023 (R$),2025 (R$),variação (R$),% do aumento total
Valor passagens,1061.10,1146.50,85.40,24.60
Valor diárias,1857.10,2117.70,260.60,75.20
Valor outros gastos,15.40,14.60,-0.80,-0.20
Valor devolução,12.90,11.60,1.30,0.40
Valor Total,2920.70,3267.20,346.60,100.00


De 2023 para 2025, a viagem média ficou R$ 347 mais cara (de R$ 2.921 para R$ 3.267). Desse aumento, 75% veio das diárias (+R$ 261) e 25% das passagens (+R$ 85). Outros gastos e devolução quase não mudaram. Em 2026, até junho, acontece o contrário: a passagem média subiu 15% em relação a 2025, e a diária caiu. Como o ano ainda não fechou, fica só como observação.

#### Parte 2: viagens mais longas ou diária mais cara?

In [ ]:
# Só viagens que receberam diárias
com_diaria = com_gasto[com_gasto["Valor diárias"] > 0].assign(
    dias=lambda d: (d["Período - Data de fim"] - d["Período - Data de início"]).dt.days + 1
)

duracao = com_diaria.groupby("ano").agg(
    dias_medios=("dias", "mean"),
    total_diarias=("Valor diárias", "sum"),
    total_dias=("dias", "sum"),
)
duracao["diária por dia (R$)"] = duracao["total_diarias"] / duracao["total_dias"]
display(duracao[["dias_medios", "diária por dia (R$)"]].round(1))

,dias_medios,diária por dia (R$)
ano,,
2023,7.00,283.60
2024,6.80,326.40
2025,6.70,335.70
2026,5.60,336.60


As viagens não ficaram mais longas, até encurtaram: de 7,0 para 6,7 dias em média entre 2023 e 2025. O que subiu foi o valor de cada dia: a diária por dia passou de R$ 284 para R$ 336 (+18%). Quase todo o aumento aconteceu de 2023 para 2024 (+15%), e depois o valor ficou estável. Um salto concentrado num ano e depois estável parece um reajuste no valor das diárias, mas isso precisa ser confirmado fora destes dados.

**Conclusão 6.7:** de 2023 para 2025, a viagem média ficou R$ 347 mais cara. 3/4 desse aumento vieram das diárias e 1/4 das passagens. E as diárias não subiram porque as viagens ficaram mais longas, inclusive, até encurtaram, mas porque cada dia de viagem ficou 18% mais caro, com quase todo o salto de 2023 para 2024.

Juntando com a 6.1: o gasto total subiu 9% sem aumento no número de viagens, e a maior parte desse aumento se explica pelo valor da diária por dia.

## 7 - CONCLUSÕES

**1. O gasto subiu, mas não porque o governo viajou mais.** De 2023 para 2025, o gasto com viagens passou de R$ 2,28 bi para R$ 2,49 bi (+9,3%). O número de viagens até caiu um pouco (−1,4%). O que subiu foi o custo de cada viagem: +11,9%, de R$ 2.921 para R$ 3.267.

**2. O que encareceu foi a diária.** Três quartos do aumento no custo por viagem vieram das diárias. E não porque as viagens ficaram mais longas (elas até encurtaram, de 7,0 para 6,7 dias), mas porque cada dia ficou 18% mais caro, com quase todo o salto de 2023 para 2024. Parece um reajuste no valor das diárias, mas isso precisa ser confirmado fora destes dados.

**3. Em 2026, o gasto está caindo, mas menos do que parece.** De janeiro a junho, o gasto ficou 7,8% abaixo de 2025. Comparando de janeiro a agosto, a queda parecia de 15,6%, mas julho e agosto ainda não foram totalmente lançados. Esse número precisa ser revisto quando o ano fechar.

**4. O gasto é concentrado.** De 35 órgãos, 9 ou 10 somam 80% do gasto, e só o Ministério da Justiça e Segurança Pública fica com cerca de um terço. Nos destinos, Brasília sozinha responde por cerca de 14% de todo o gasto. As viagens internacionais são só 3% do total, mas ficam com 13% a 15% do gasto.

**5. "Urgente" não quer dizer emergência.** Quase 60% das viagens são marcadas como urgentes, e elas não custam mais que as normais. O mais provável é que a marcação indique pedido feito com pouca antecedência.

### Limites da análise

- Só 38% do gasto tem cargo identificado, e 85% das viagens têm destino identificado. O resto está em sigilo ou não foi informado.
- Os dados de 2026 vão até junho de forma confiável.
- Nas viagens com mais de uma cidade, dividi o valor igualmente entre elas, porque os dados não dizem quanto foi gasto em cada uma.
- O significado da marcação de urgência e o reajuste das diárias são hipóteses que ainda precisam ser confirmadas.

### Próximos passos

- Atualizar 2026 quando o Portal publicar os meses que faltam.
- Usar o arquivo de trechos do Portal para entender as viagens com mais de uma cidade.
- Comparar o custo de viagens urgentes e normais para o mesmo destino.
- Identificar os militares pela patente, para incluir parte do gasto sem cargo na análise por cargo.
- Atualizar esse estudo anualmente, para acompanhar a como os gastos evoluem.
